# 02 — Preparing the modelling dataset

## Objective

Create one row per eligible order to predict late delivery
at the moment the order is approved.

Only information available at approval may be used as model inputs.
Actual delivery dates are used to define the outcome, not as inputs.

This notebook will:
1. Reload the original data independently.
2. Define eligibility and document exclusions.
3. Build and validate order-level inputs.
4. Separate earlier training orders from later evaluation orders.

The earlier exploration covered the available period.
The evaluation period will not be used for model tuning.

## 1. Load the source data

Reload the original CSV files so this notebook runs independently.
Preserve postal codes as text to retain leading zeros.

In [1]:
from pathlib import Path
import pandas as pd

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

raw_dir = project_dir / "data" / "raw"

orders = pd.read_csv(raw_dir / "olist_orders_dataset.csv")
items = pd.read_csv(raw_dir / "olist_order_items_dataset.csv")

customers = pd.read_csv(
    raw_dir / "olist_customers_dataset.csv",
    dtype={"customer_zip_code_prefix": "string"}
)

sellers = pd.read_csv(
    raw_dir / "olist_sellers_dataset.csv",
    dtype={"seller_zip_code_prefix": "string"}
)

products = pd.read_csv(raw_dir / "olist_products_dataset.csv")

tables = {
    "orders": orders,
    "items": items,
    "customers": customers,
    "sellers": sellers,
    "products": products
}

pd.DataFrame([
    {"table": name, "rows": len(table), "columns": table.shape[1]}
    for name, table in tables.items()
])

,table,rows,columns
0,orders,99441,8
1,items,112650,7
2,customers,99441,5
3,sellers,3095,4
4,products,32951,9


## 2. Validate dates and table identifiers

Convert order timestamps into datetime values.
Stop if a non-missing timestamp cannot be parsed.

Check that table keys are complete and unique before joining.
Repeated order IDs in the items table are expected; the combination
of order_id and order_item_id must be unique.

In [2]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        format="%Y-%m-%d %H:%M:%S",
        errors="raise"
    )

key_checks = [
    ("orders", orders, ["order_id"]),
    ("customers", customers, ["customer_id"]),
    ("sellers", sellers, ["seller_id"]),
    ("products", products, ["product_id"]),
    ("items", items, ["order_id", "order_item_id"])
]

for name, table, keys in key_checks:
    assert not table[keys].isna().any().any(), (
        f"{name}: missing key values"
    )
    assert not table.duplicated(keys).any(), (
        f"{name}: duplicate keys"
    )

print("Date conversion passed.")
print("All five table-key checks passed.")
print(
    "Orders missing approval dates:",
    orders["order_approved_at"].isna().sum()
)

Date conversion passed.
All five table-key checks passed.
Orders missing approval dates: 160


## 3. Define modelling eligibility

For the first model, use orders purchased from January 2017 through
August 2018, where the dataset has substantial monthly coverage.

Require:
- Delivered status.
- Recorded purchase, approval, estimated delivery, and actual delivery dates.
- Approval at or after purchase.
- Actual delivery at or after approval.
- A promised delivery calendar date on or after approval.

Count exclusions sequentially so each excluded order is counted once.

This model will be evaluated on eligible completed deliveries.
It does not predict cancellations or resolve unknown delivery outcomes.
Carrier timestamp anomalies alone do not trigger exclusion because
carrier timestamps will not be used as prediction inputs.

In [3]:
rules = [
    (
        "Purchased Jan 2017–Aug 2018",
        orders["order_purchase_timestamp"].ge("2017-01-01")
        & orders["order_purchase_timestamp"].lt("2018-09-01")
    ),
    ("Delivered status", orders["order_status"].eq("delivered")),
    (
        "Required dates present",
        orders[
            [
                "order_purchase_timestamp",
                "order_approved_at",
                "order_estimated_delivery_date",
                "order_delivered_customer_date"
            ]
        ].notna().all(axis=1)
    ),
    (
        "Approval at or after purchase",
        orders["order_approved_at"]
        >= orders["order_purchase_timestamp"]
    ),
    (
        "Delivery at or after approval",
        orders["order_delivered_customer_date"]
        >= orders["order_approved_at"]
    ),
    (
        "Promised date on or after approval date",
        orders["order_estimated_delivery_date"].dt.normalize()
        >= orders["order_approved_at"].dt.normalize()
    )
]

keep = pd.Series(True, index=orders.index)
exclusion_log = []

for rule_name, passes in rules:
    before = int(keep.sum())
    keep = keep & passes.fillna(False)

    exclusion_log.append({
        "rule": rule_name,
        "excluded_at_this_step": before - int(keep.sum()),
        "orders_remaining": int(keep.sum())
    })

model_orders = orders.loc[keep].copy()

assert (
    sum(row["excluded_at_this_step"] for row in exclusion_log)
    + len(model_orders)
    == len(orders)
)

pd.DataFrame(exclusion_log)

,rule,excluded_at_this_step,orders_remaining
0,Purchased Jan 2017–Aug 2018,349,99092
1,Delivered status,2881,96211
2,Required dates present,22,96189
3,Approval at or after purchase,0,96189
4,Delivery at or after approval,61,96128
5,Promised date on or after approval date,1,96127


## 4. Define the prediction target

After applying the eligibility rules, 96,127 orders remain.

An order is late if its actual delivery calendar date is after
its promised delivery date. Arrival on the promised day counts
as on time.

Target: is_late
- 1 = late
- 0 = on time or early

Actual delivery dates are used only to define the target and
check when outcomes became known. They are not model inputs.

In [5]:
actual_day = (
    model_orders["order_delivered_customer_date"].dt.normalize()
)

promised_day = (
    model_orders["order_estimated_delivery_date"].dt.normalize()
)

model_orders["is_late"] = (
    actual_day > promised_day
).astype("int8")

target_summary = (
    model_orders["is_late"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("is_late")
    .to_frame("orders")
)

target_summary["percent"] = (
    target_summary["orders"] / len(model_orders) * 100
).round(2)

target_summary

,orders,percent
is_late,,
0,89598,93.21
1,6529,6.79


### Target distribution

The modelling dataset contains 96,127 orders:
- 89,598 (93.21%) were on time or early.
- 6,529 (6.79%) were late.

Late orders are the minority class. Accuracy alone would be
misleading: always predicting “on time” would identify no late orders.

## 5. Add customer location

Attach customer state and postal-code prefix to each eligible order.
Verify that every order matches a customer and no rows are multiplied.

In [6]:
model_data = model_orders.merge(
    customers[
        ["customer_id", "customer_state", "customer_zip_code_prefix"]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator="customer_match"
)

assert len(model_data) == len(model_orders)
assert model_data["order_id"].is_unique
assert model_data["customer_match"].eq("both").all()

print(f"Orders after customer join: {len(model_data):,}")

model_data[
    ["customer_state", "customer_zip_code_prefix"]
].isna().sum().rename("missing_values").to_frame()

Orders after customer join: 96,127


,missing_values
customer_state,0
customer_zip_code_prefix,0


## 6. Add order-item summaries

Aggregate items into one row per order, recording item count,
distinct product and seller counts, total item price, and freight.

Check amounts before summing and verify that every modelling
order matches an item summary. Item price totals exclude freight.

In [7]:
# Check fields needed for aggregation
required = [
    "order_id", "order_item_id", "product_id",
    "seller_id", "price", "freight_value"
]

assert items[required].notna().all().all()
assert items[["price", "freight_value"]].ge(0).all().all()

order_items_summary = (
    items.groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "size"),
        distinct_products=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        total_item_price=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
)

assert order_items_summary["item_count"].sum() == len(items)

# Use a new table so rerunning this cell does not repeat the join
model_with_items = model_data.merge(
    order_items_summary,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator="items_match"
)

assert len(model_with_items) == len(model_data)
assert model_with_items["items_match"].eq("both").all()

print(f"Orders after item join: {len(model_with_items):,}")

model_with_items[
    ["item_count", "distinct_products", "seller_count",
     "total_item_price", "total_freight"]
].isna().sum().rename("missing_values").to_frame()

Orders after item join: 96,127


,missing_values
item_count,0
distinct_products,0
seller_count,0
total_item_price,0
total_freight,0


## 7. Add seller–customer state relationship

Match each item to its seller and the order's customer state.
Then classify each order as:
- All sellers in the customer's state.
- All sellers outside the customer's state.
- Mixed seller locations.

Seller locations are catalog snapshots. Using them at approval
assumes those locations also applied when the order was placed.

In [8]:
item_locations = items.merge(
    sellers[["seller_id", "seller_state"]],
    on="seller_id",
    how="left",
    validate="many_to_one",
    indicator="seller_match"
)

assert len(item_locations) == len(items)
assert item_locations["seller_match"].eq("both").all()
assert item_locations["seller_state"].notna().all()

# Keep items belonging to our modelling orders
model_item_locations = item_locations.merge(
    model_data[["order_id", "customer_state"]],
    on="order_id",
    how="inner",
    validate="many_to_one"
)

model_item_locations["different_state"] = (
    model_item_locations["seller_state"]
    != model_item_locations["customer_state"]
)

routes = (
    model_item_locations.groupby("order_id")["different_state"]
    .agg(any_outside="any", all_outside="all")
    .reset_index()
)

routes["route_group"] = "Mixed seller locations"
routes.loc[~routes["any_outside"], "route_group"] = "All same state"
routes.loc[routes["all_outside"], "route_group"] = "All outside state"

model_with_routes = model_with_items.merge(
    routes[["order_id", "route_group"]],
    on="order_id",
    how="left",
    validate="one_to_one"
)

assert len(model_with_routes) == len(model_with_items)
assert model_with_routes["route_group"].notna().all()

model_with_routes["route_group"].value_counts().to_frame("orders")

,orders
route_group,
All outside state,61451
All same state,34430
Mixed seller locations,246


## 8. Prepare product measurements

Treat zero or negative measurements as unknown.
Attach product weight and dimensions to each modelling item.

Catalog measurements are assumed to represent the product at
order approval; historical catalog versions are unavailable.
Missing values will be handled using training data only.

In [9]:
measurement_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

product_measurements = products[
    ["product_id"] + measurement_columns
].copy()

# Preserve missing values and replace non-positive measurements
for column in measurement_columns:
    product_measurements[column] = (
        product_measurements[column]
        .where(product_measurements[column] > 0)
    )

product_measurements["product_volume_cm3"] = (
    product_measurements["product_length_cm"]
    * product_measurements["product_height_cm"]
    * product_measurements["product_width_cm"]
)

model_items = model_item_locations.merge(
    product_measurements,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="product_match"
)

assert len(model_items) == len(model_item_locations)
assert model_items["product_match"].eq("both").all()

print(f"Modelling item rows: {len(model_items):,}")

model_items[
    ["product_weight_g", "product_volume_cm3"]
].isna().sum().rename("items_with_unknown_measurement").to_frame()

Modelling item rows: 109,787


,items_with_unknown_measurement
product_weight_g,26
product_volume_cm3,18


### Aggregate product measurements by order

Calculate total catalog weight and summed product volume per order.
If any item's weight or volume is unknown, leave the corresponding
order total missing.

Summed product volume is not shipping-package volume.

In [10]:
order_measurements = (
    model_items.groupby("order_id", as_index=False)
    .agg(
        measurement_item_count=("order_item_id", "size"),
        known_weight_items=("product_weight_g", "count"),
        known_volume_items=("product_volume_cm3", "count"),
        total_weight_g=("product_weight_g", "sum"),
        total_product_volume_cm3=("product_volume_cm3", "sum")
    )
)

incomplete_weight = (
    order_measurements["known_weight_items"]
    < order_measurements["measurement_item_count"]
)

incomplete_volume = (
    order_measurements["known_volume_items"]
    < order_measurements["measurement_item_count"]
)

order_measurements.loc[incomplete_weight, "total_weight_g"] = float("nan")
order_measurements.loc[
    incomplete_volume, "total_product_volume_cm3"
] = float("nan")

model_with_products = model_with_routes.merge(
    order_measurements,
    on="order_id",
    how="left",
    validate="one_to_one"
)

assert len(model_with_products) == len(model_with_routes)
assert (
    model_with_products["measurement_item_count"]
    == model_with_products["item_count"]
).all()

print(f"Orders after product aggregation: {len(model_with_products):,}")

model_with_products[
    ["total_weight_g", "total_product_volume_cm3"]
].isna().sum().rename("orders_with_unknown_total").to_frame()

Orders after product aggregation: 96,127


,orders_with_unknown_total
total_weight_g,22
total_product_volume_cm3,16


## 9. Create time-based inputs available at approval

Create:
- Hours from purchase to approval.
- Calendar days from approval to the promised delivery date.
- Approval month, weekday, and hour.

Assume the recorded estimated delivery date was available at approval.
The dataset does not provide a history of revisions to that promise.

Actual delivery and carrier handover timestamps are not used
to create these inputs.

In [11]:
prepared_orders = model_with_products.copy()

approved = prepared_orders["order_approved_at"]
purchased = prepared_orders["order_purchase_timestamp"]
promised = prepared_orders["order_estimated_delivery_date"]

prepared_orders["approval_delay_hours"] = (
    approved - purchased
).dt.total_seconds() / 3600

prepared_orders["promised_days_from_approval"] = (
    promised.dt.normalize() - approved.dt.normalize()
).dt.days

prepared_orders["approval_month"] = approved.dt.month
prepared_orders["approval_weekday"] = approved.dt.dayofweek
prepared_orders["approval_hour"] = approved.dt.hour

time_features = [
    "approval_delay_hours",
    "promised_days_from_approval",
    "approval_month",
    "approval_weekday",
    "approval_hour"
]

assert prepared_orders[time_features].notna().all().all()
assert prepared_orders["approval_delay_hours"].ge(0).all()
assert prepared_orders["promised_days_from_approval"].ge(0).all()

prepared_orders[time_features].describe().round(2)

/var/folders/vm/wpy4jwz9593d3_cf75gx3y180000gn/T/ipykernel_92212/489735488.py:16: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  prepared_orders["approval_weekday"] = approved.dt.dayofweek


,approval_delay_hours,promised_days_from_approval,approval_month,approval_weekday,approval_hour
count,96127.00,96127.00,96127.00,96127.00,96127.00
mean,10.17,23.78,6.03,2.73,12.73
std,20.16,8.61,3.23,1.87,6.61
min,0.00,1.00,1.00,0.00,0.00
25%,0.22,18.00,3.00,1.00,8.00
50%,0.34,23.00,6.00,3.00,14.00
75%,14.36,29.00,8.00,4.00,18.00
max,741.44,154.00,12.00,6.00,23.00


### Interpretation of time-based inputs

All five features have values for all 96,127 eligible orders.

The median approval delay was 0.34 hours (approximately 20 minutes),
while the average was 10.17 hours. Some long approval delays pull
the average upward. The maximum was 741.44 hours, or about 31 days.

The median promised delivery window after approval was 23 calendar
days. The middle 50% of orders had windows between 18 and 29 days.
The maximum of 154 days is unusual and remains documented for review.

Approval months, weekdays, and hours fall within their expected
ranges. Their averages have limited practical meaning because
these values represent positions within calendar cycles.

These checks confirm completeness and basic validity; they do not
yet show how useful the features will be for predicting delays.

## 10. Select model inputs

Use an explicit list of inputs available or assumed available
at approval.

Exclude actual delivery dates, carrier timestamps, final order
status, identifiers, and the target from model inputs.

Calendar fields are treated as categories for the first baseline.
Missing measurements will be handled after splitting the data,
using training data only.

In [12]:
numeric_features = [
    "item_count",
    "distinct_products",
    "seller_count",
    "total_item_price",
    "total_freight",
    "total_weight_g",
    "total_product_volume_cm3",
    "approval_delay_hours",
    "promised_days_from_approval"
]

categorical_features = [
    "customer_state",
    "route_group",
    "approval_month",
    "approval_weekday",
    "approval_hour"
]

feature_columns = numeric_features + categorical_features

X = prepared_orders[feature_columns].copy()
y = prepared_orders["is_late"].copy()

assert "is_late" not in X.columns
assert X.index.equals(y.index)

print(f"Orders: {len(X):,}")
print(f"Model inputs: {X.shape[1]}")
print(f"Numeric inputs: {len(numeric_features)}")
print(f"Categorical inputs: {len(categorical_features)}")

X.isna().sum().rename("missing_values").to_frame()

Orders: 96,127
Model inputs: 14
Numeric inputs: 9
Categorical inputs: 5


,missing_values
item_count,0
distinct_products,0
seller_count,0
total_item_price,0
total_freight,0
total_weight_g,22
total_product_volume_cm3,16
approval_delay_hours,0
promised_days_from_approval,0
customer_state,0


### Input selection results

The input table contains 96,127 orders and 14 features:
nine numeric and five categorical.

Total weight is missing for 22 orders, and total product volume
is missing for 16. These groups may overlap. All other inputs
are complete.

Missing values remain unfilled until after the chronological split.
Any imputation values will be learned from training data only.

## 11. Define chronological splits

Split by approval timestamp:
- Training: before 1 May 2018.
- Validation: 1 May–30 June 2018.
- Test: 1 July 2018 onward within the eligible dataset.

Training outcomes must be recorded before 1 May.
Validation outcomes must be recorded before 1 July, when model
selection would finish.

This conservative rule uses recorded delivery as the time the
outcome becomes known. It may exclude unusually slow deliveries;
we will report those exclusions.

The test period was included in earlier descriptive exploration.
It is reserved from model tuning, but is not completely unseen.

In [13]:
validation_start = pd.Timestamp("2018-05-01")
test_start = pd.Timestamp("2018-07-01")

approval = prepared_orders["order_approved_at"]
delivery = prepared_orders["order_delivered_customer_date"]

train_period = approval < validation_start
validation_period = (
    (approval >= validation_start) & (approval < test_start)
)
test_mask = approval >= test_start

train_mask = train_period & (delivery < validation_start)
validation_mask = validation_period & (delivery < test_start)

split_summary = pd.DataFrame({
    "period_orders": [
        train_period.sum(),
        validation_period.sum(),
        test_mask.sum()
    ],
    "retained_orders": [
        train_mask.sum(),
        validation_mask.sum(),
        test_mask.sum()
    ]
}, index=["Training", "Validation", "Test"])

split_summary["excluded_pending_outcome"] = (
    split_summary["period_orders"]
    - split_summary["retained_orders"]
)

assert split_summary["period_orders"].sum() == len(prepared_orders)

split_summary

,period_orders,retained_orders,excluded_pending_outcome
Training,70577,68010,2567
Validation,13034,11381,1653
Test,12516,12516,0


### Chronological split results

- Training: 68,010 orders retained; 2,567 excluded because their
  deliveries were not recorded before the validation cutoff.
- Validation: 11,381 retained; 1,653 excluded because their
  deliveries were not recorded before the test cutoff.
- Test: 12,516 orders retained from the already eligible dataset.

The outcome cutoff removes approximately 3.6% of training-period
orders and 12.7% of validation-period orders. This may favour
faster deliveries and must be considered when interpreting results.

Zero additional test exclusions does not mean every original order
has an observed outcome: the initial eligibility rules already
restricted the dataset to completed deliveries.

In [14]:
X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_validation = X.loc[validation_mask].copy()
y_validation = y.loc[validation_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

assert set(X_train.index).isdisjoint(X_validation.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_validation.index).isdisjoint(X_test.index)

# Inspect development data only; reserve test outcomes for evaluation
development_summary = pd.DataFrame({
    "orders": [len(y_train), len(y_validation)],
    "late_orders": [y_train.sum(), y_validation.sum()],
    "late_rate_percent": [
        y_train.mean() * 100,
        y_validation.mean() * 100
    ]
}, index=["Training", "Validation"])

development_summary.round(2)

,orders,late_orders,late_rate_percent
Training,68010,4944,7.27
Validation,11381,444,3.90


### Training and validation target distributions

Training contains 68,010 orders, including 4,944 late deliveries
(7.27%). Validation contains 11,381 orders, including 444 late
deliveries (3.90%).

This difference may reflect changes over time and the exclusion
of outcomes not recorded by each cutoff. It cannot be attributed
to either explanation from this comparison alone.

Model performance will be compared with simple baselines on
the same validation orders. Test outcomes remain reserved.

In [ ]:
%pip install scikit-learn

## 12. Establish a simple baseline

Assign every validation order the training late-delivery rate.
This baseline ignores order characteristics and cannot rank risks.

At a 50% decision threshold, it predicts every order as on time.
Measure accuracy and recall to show why accuracy alone is misleading.

Use validation data only; leave the test set untouched.

In [16]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    average_precision_score
)

baseline_probability = float(y_train.mean())

baseline_scores = np.full(
    len(y_validation), baseline_probability
)
baseline_predictions = (baseline_scores >= 0.5).astype(int)

baseline_results = pd.DataFrame([{
    "model": "Constant training-rate baseline",
    "predicted_late_probability": baseline_probability,
    "accuracy": accuracy_score(y_validation, baseline_predictions),
    "late_order_recall": recall_score(
        y_validation, baseline_predictions, zero_division=0
    ),
    "average_precision": average_precision_score(
        y_validation, baseline_scores
    )
}])

baseline_results.round(4)

,model,predicted_late_probability,accuracy,late_order_recall,average_precision
0,Constant training-rate baseline,0.0727,0.961,0.0,0.039


## 13. Train logistic regression

Fill missing numeric inputs with training medians, scale numeric
features, and encode categories as indicator columns.

Fit preprocessing and the classifier on training data only.
Validation and test data do not influence this fitting step.

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    )
])

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000))
])

logistic_model.fit(X_train, y_train)

print("Logistic regression training completed.")
print(f"Training orders: {len(X_train):,}")

Logistic regression training completed.
Training orders: 68,010


## 14. Check how well the model works

Compare our model with the simple baseline using validation orders
that were not used to train the model.

- Accuracy: how often the prediction is correct.
- Recall: how many of the actual late orders we catch.
- Average precision: how well the model puts late orders near
  the top of the risk list. Higher is better.

For now, we predict “late” when the model gives a risk of 50% or more.

We are saving the test data for the final evaluation.

In [18]:
validation_scores = logistic_model.predict_proba(
    X_validation
)[:, 1]

validation_predictions = (
    validation_scores >= 0.5
).astype(int)

logistic_results = pd.DataFrame([{
    "model": "Logistic regression",
    "predicted_late_probability": validation_scores.mean(),
    "accuracy": accuracy_score(
        y_validation, validation_predictions
    ),
    "late_order_recall": recall_score(
        y_validation, validation_predictions, zero_division=0
    ),
    "average_precision": average_precision_score(
        y_validation, validation_scores
    )
}])

comparison = pd.concat(
    [baseline_results, logistic_results],
    ignore_index=True
).rename(columns={
    "predicted_late_probability": "mean_predicted_late_probability"
})

comparison.round(4)

,model,mean_predicted_late_probability,accuracy,late_order_recall,average_precision
0,Constant training-rate baseline,0.0727,0.961,0.0,0.039
1,Logistic regression,0.0255,0.961,0.0,0.117


### What the results mean

The model is better than the baseline at putting late orders
near the top of the risk list. Average precision increased
from 0.039 to 0.117.

However, using a 50% risk cutoff catches none of the late orders.
Both models have 96.1% accuracy, showing why accuracy alone
does not tell us whether the model is useful.

The model's average predicted risk is 2.55%, compared with
an actual late rate of 3.90%. It underestimates the overall
late rate in this validation period.

Next, we will check how many late orders appear in the
highest-risk 10% of validation orders.

## 15. Check the highest-risk 10% of orders

Rank validation orders by predicted risk and select the top 10%.

Measure:
- Precision: what percentage of selected orders were actually late?
- Recall: what percentage of all late orders did we catch?

This checks a fixed review budget across the validation period.
It does not yet simulate a daily review queue.

In [19]:
risk_ranking = pd.DataFrame({
    "order_id": prepared_orders.loc[X_validation.index, "order_id"],
    "risk_score": validation_scores,
    "is_late": y_validation
})

# Order ID provides a consistent tie-breaker for identical scores
risk_ranking = risk_ranking.sort_values(
    ["risk_score", "order_id"],
    ascending=[False, True]
)

review_count = int(np.ceil(len(risk_ranking) * 0.10))
selected = risk_ranking.head(review_count)

late_caught = int(selected["is_late"].sum())
total_late = int(y_validation.sum())

precision = late_caught / review_count
recall = late_caught / total_late
random_selection_precision = float(y_validation.mean())

print(f"Orders selected for review: {review_count:,}")
print(f"Late orders caught: {late_caught:,} of {total_late:,}")
print(f"Precision: {precision:.2%}")
print(f"Recall: {recall:.2%}")
print(
    "Expected precision from random selection: "
    f"{random_selection_precision:.2%}"
)
print(
    "Lift over random selection: "
    f"{precision / random_selection_precision:.2f}x"
)

Orders selected for review: 1,139
Late orders caught: 163 of 444
Precision: 14.31%
Recall: 36.71%
Expected precision from random selection: 3.90%
Lift over random selection: 3.67x


### What the top-10% results mean

Reviewing the 1,139 highest-risk orders caught 163 of the
444 late deliveries: 36.71% of all late orders.

Of the selected orders, 14.31% were actually late.
Random selection would have an expected late rate of 3.90%,
so the model concentrated late orders by approximately 3.67 times.

There are trade-offs: 976 selected orders were on time, and
281 late orders were missed.

These results suggest the model could help prioritise reviews.
They do not show that reviewing an order prevents a delay.
The results apply to our filtered validation data and a review
budget across the whole period, not yet a daily queue.

## 16. Test a daily review queue

For each approval day, select up to 50 orders with the highest
predicted risk. If fewer than 50 orders exist, select them all.

Measure how many orders staff would review and how many late
deliveries would be identified.

This is a retrospective simulation using eligible validation orders,
not a live test or evidence that reviews prevent delays.

In [20]:
daily_ranking = risk_ranking.copy()

daily_ranking["approval_day"] = prepared_orders.loc[
    daily_ranking.index, "order_approved_at"
].dt.normalize()

daily_selected = (
    daily_ranking.sort_values(
        ["approval_day", "risk_score", "order_id"],
        ascending=[True, False, True]
    )
    .groupby("approval_day")
    .head(50)
)

reviewed = len(daily_selected)
late_caught = int(daily_selected["is_late"].sum())
total_late = int(daily_ranking["is_late"].sum())

print(f"Days represented: {daily_ranking['approval_day'].nunique()}")
print(f"Orders reviewed: {reviewed:,}")
print(f"Share of validation orders reviewed: {reviewed / len(daily_ranking):.2%}")
print(f"Late orders caught: {late_caught:,} of {total_late:,}")
print(f"Precision: {late_caught / reviewed:.2%}")
print(f"Recall: {late_caught / total_late:.2%}")

Days represented: 60
Orders reviewed: 2,913
Share of validation orders reviewed: 25.60%
Late orders caught: 185 of 444
Precision: 6.35%
Recall: 41.67%


### Daily queue results

Reviewing up to 50 orders per approval day selected 2,913 orders
(25.60% of validation orders) and caught 185 of 444 late deliveries.

Recall was 41.67%, and precision was 6.35%.

Compared with selecting the top 10% across the entire period,
the daily queue required 1,774 additional reviews to catch
22 additional late orders.

The approaches have different budgets and timing constraints,
so this is a workload comparison rather than a like-for-like test.

Daily results also inherit the validation exclusions: orders
without a recorded delivery before July were removed. This
limits how closely the simulation represents a real incoming queue.

## 17. Compare with random daily selection

Compare the model with randomly selecting the same number of
orders each day.

Calculate the expected number of late orders caught by random
selection, using each day's late rate. No repeated simulation
is needed for this expected value.

In [21]:
daily_totals = (
    daily_ranking.groupby("approval_day")
    .agg(
        orders=("is_late", "size"),
        late_orders=("is_late", "sum")
    )
)

daily_totals["review_budget"] = daily_totals["orders"].clip(upper=50)

expected_random_caught = (
    daily_totals["review_budget"]
    * daily_totals["late_orders"]
    / daily_totals["orders"]
).sum()

model_caught = int(daily_selected["is_late"].sum())
reviewed = len(daily_selected)
total_late = int(daily_ranking["is_late"].sum())

daily_comparison = pd.DataFrame({
    "approach": ["Random selection (expected)", "Model ranking"],
    "orders_reviewed": [reviewed, reviewed],
    "late_orders_caught": [expected_random_caught, model_caught],
    "precision_percent": [
        expected_random_caught / reviewed * 100,
        model_caught / reviewed * 100
    ],
    "recall_percent": [
        expected_random_caught / total_late * 100,
        model_caught / total_late * 100
    ]
})

print(
    "Model lift over random daily selection: "
    f"{model_caught / expected_random_caught:.2f}x"
)

daily_comparison.round(2)

Model lift over random daily selection: 2.10x


,approach,orders_reviewed,late_orders_caught,precision_percent,recall_percent
0,Random selection (expected),2913,88.01,3.02,19.82
1,Model ranking,2913,185.00,6.35,41.67


### Final baseline finding

Both approaches review 2,913 orders using the same daily budget.

The model catches 185 late orders, compared with approximately
88 expected from random selection: a lift of 2.10 times.

Recall improves from an expected 19.82% to 41.67%.
Precision improves from an expected 3.02% to 6.35%.

This supports using the model to prioritise reviews, although most
flagged orders are still on time. Results apply to the filtered
validation dataset and do not show that reviews prevent delays.

## 18. Compare a boosted-tree model

Train a second model that can learn more complex relationships
between order characteristics and delivery risk.

Use the same training data, validation orders, and daily review
budget as logistic regression. Keep the test set reserved.

In [22]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder

# Keep numeric values; convert categories into category codes
tree_preprocessor = ColumnTransformer([
    ("numeric", "passthrough", numeric_features),
    (
        "categorical",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=np.nan
        ),
        categorical_features
    )
])

# Tell the model which transformed columns represent categories
category_positions = list(range(
    len(numeric_features),
    len(numeric_features) + len(categorical_features)
))

boosted_model = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "classifier",
        HistGradientBoostingClassifier(
            categorical_features=category_positions,
            learning_rate=0.1,
            max_iter=150,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            early_stopping=False,
            random_state=42
        )
    )
])

boosted_model.fit(X_train, y_train)

print("Boosted-tree training completed.")
print(f"Training orders: {len(X_train):,}")

Boosted-tree training completed.
Training orders: 68,010


### Compare both models fairly

For each model, select up to 50 highest-risk orders per approval day.

Compare how many late orders each model catches, its precision,
recall, and average precision. Both models use the same validation
orders and review budget. The test set remains reserved.

In [24]:
boosted_validation_scores = boosted_model.predict_proba(
    X_validation
)[:, 1]

def evaluate_daily_queue(model_name, scores):
    ranked = pd.DataFrame({
        "order_id": prepared_orders.loc[X_validation.index, "order_id"],
        "approval_day": prepared_orders.loc[
            X_validation.index, "order_approved_at"
        ].dt.normalize(),
        "risk_score": scores,
        "is_late": y_validation
    })

    selected = (
        ranked.sort_values(
            ["approval_day", "risk_score", "order_id"],
            ascending=[True, False, True]
        )
        .groupby("approval_day")
        .head(50)
    )

    caught = int(selected["is_late"].sum())

    return {
        "model": model_name,
        "orders_reviewed": len(selected),
        "late_orders_caught": caught,
        "precision_percent": caught / len(selected) * 100,
        "recall_percent": caught / y_validation.sum() * 100,
        "average_precision": average_precision_score(
            y_validation, scores
        )
    }

model_comparison = pd.DataFrame([
    evaluate_daily_queue("Logistic regression", validation_scores),
    evaluate_daily_queue("Boosted trees", boosted_validation_scores)
])

model_comparison.round(4)

,model,orders_reviewed,late_orders_caught,precision_percent,recall_percent,average_precision
0,Logistic regression,2913,185,6.3508,41.6667,0.117
1,Boosted trees,2913,141,4.8404,31.7568,0.083


### Model comparison and selection

Logistic regression caught 185 late orders, compared with 141
for boosted trees, using the same daily budget and 2,913 reviews.

Logistic regression achieved:
- Precision: 6.35% versus 4.84%.
- Recall: 41.67% versus 31.76%.
- Average precision: 0.117 versus 0.083.

We select logistic regression for the first project version because
it performed better on validation data and is simpler to explain.

This result applies to the models and settings tested. It does not
mean logistic regression always outperforms boosted trees.

The selected model and daily review budget are now fixed before
evaluating the reserved test data.

## 19. Final test evaluation

Evaluate the selected logistic regression model on later orders.
Keep the model unchanged and review up to 50 orders per approval day.

Compare its results with random selection using the same daily budget.
Do not use these results to tune the model.

The test period appeared in earlier exploration, and results apply
only to eligible completed deliveries. Unlike training and validation,
no additional delivery-outcome cutoff was applied to the test set.

In [25]:
test_scores = logistic_model.predict_proba(X_test)[:, 1]

test_ranking = pd.DataFrame({
    "order_id": prepared_orders.loc[X_test.index, "order_id"],
    "approval_day": prepared_orders.loc[
        X_test.index, "order_approved_at"
    ].dt.normalize(),
    "risk_score": test_scores,
    "is_late": y_test
})

test_selected = (
    test_ranking.sort_values(
        ["approval_day", "risk_score", "order_id"],
        ascending=[True, False, True]
    )
    .groupby("approval_day")
    .head(50)
)

test_daily = (
    test_ranking.groupby("approval_day")
    .agg(
        orders=("is_late", "size"),
        late_orders=("is_late", "sum")
    )
)

test_daily["budget"] = test_daily["orders"].clip(upper=50)

random_expected = (
    test_daily["budget"]
    * test_daily["late_orders"]
    / test_daily["orders"]
).sum()

caught = int(test_selected["is_late"].sum())
reviewed = len(test_selected)
total_late = int(y_test.sum())

print(f"Test orders: {len(y_test):,}")
print(f"Actual late orders: {total_late:,}")
print(f"Actual late rate: {y_test.mean():.2%}")
print(f"Orders reviewed: {reviewed:,} ({reviewed / len(y_test):.2%})")
print(f"Late orders caught: {caught:,}")
print(f"Precision: {caught / reviewed:.2%}")
print(f"Recall: {caught / total_late:.2%}")
print(f"Expected random late orders caught: {random_expected:.2f}")
print(f"Lift over random daily selection: {caught / random_expected:.2f}x")
print(
    "Average precision: "
    f"{average_precision_score(y_test, test_scores):.4f}"
)

Test orders: 12,516
Actual late orders: 599
Actual late rate: 4.79%
Orders reviewed: 2,774 (22.16%)
Late orders caught: 143
Precision: 5.16%
Recall: 23.87%
Expected random late orders caught: 114.45
Lift over random daily selection: 1.25x
Average precision: 0.0769


### Final test results

The test dataset contains 12,516 eligible orders, including
599 late deliveries (4.79%).

Reviewing up to 50 orders per approval day:
- Selected 2,774 orders (22.16%).
- Caught 143 late deliveries (23.87% recall).
- Achieved 5.16% precision.
- Caught approximately 29 more late orders than expected from
  random selection with the same daily budget.
- Achieved 1.25 times the expected random yield.

Average precision was 0.0769.

### Conclusion and limitations

The model offers modest prioritisation value on this test period.
Its advantage over random selection was smaller than on validation,
where daily lift was 2.10 times.

Most flagged orders were on time, and most late orders were missed.
This version is a portfolio prototype, not a deployment-ready system.

The periods have different outcome rates and exclusion rules.
Earlier exploration also included the test period. These limitations
must accompany the reported results.

No model settings will be changed based on this test result.
Reviews have not been shown to prevent delays.

## 20. Save the selected model

Save the trained logistic regression pipeline, input-column lists,
and software versions for the demo.

The saved pipeline includes preprocessing learned from training data.
It is not retrained on validation or test orders.

In [26]:
import joblib
import sklearn
import sys

models_dir = project_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

model_bundle = {
    "pipeline": logistic_model,
    "feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target": "is_late",
    "prediction_time": "order approval",
    "daily_review_budget": 50,
    "versions": {
        "python": sys.version,
        "scikit_learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__
    }
}

model_path = models_dir / "logistic_delivery_risk_v1.joblib"
joblib.dump(model_bundle, model_path)

# Verify that loading the saved model preserves its predictions
loaded_bundle = joblib.load(model_path)

np.testing.assert_allclose(
    loaded_bundle["pipeline"].predict_proba(X_test.head(20)),
    logistic_model.predict_proba(X_test.head(20))
)

print(f"Model saved: {model_path}")
print("Reload check passed: predictions match.")

Model saved: /Users/javiermorande/Desktop/olist-delivery-risk/models/logistic_delivery_risk_v1.joblib
Reload check passed: predictions match.


## 21. Audit the evaluation cutoffs

The previous split excluded orders delivered after each cutoff.
This may remove more late deliveries than on-time deliveries.

Compare retained and excluded orders within the training and
validation periods to measure that effect.

This is a retrospective audit of development data.
Do not inspect test results or change model settings in this step.

In [27]:
cutoff_audit = []

for period_name, period_mask, retained_mask in [
    ("Training", train_period, train_mask),
    ("Validation", validation_period, validation_mask)
]:
    groups = {
        "Retained": period_mask & retained_mask,
        "Excluded by outcome cutoff": period_mask & ~retained_mask
    }

    for group_name, mask in groups.items():
        outcomes = prepared_orders.loc[mask, "is_late"]

        cutoff_audit.append({
            "period": period_name,
            "group": group_name,
            "orders": len(outcomes),
            "late_orders": int(outcomes.sum()),
            "late_rate_percent": outcomes.mean() * 100
        })

pd.DataFrame(cutoff_audit).round(2)

,period,group,orders,late_orders,late_rate_percent
0,Training,Retained,68010,4944,7.27
1,Training,Excluded by outcome cutoff,2567,461,17.96
2,Validation,Retained,11381,444,3.90
3,Validation,Excluded by outcome cutoff,1653,81,4.90


### Cutoff audit findings

Orders excluded by the delivery-outcome cutoff were more likely
to be late than retained orders.

The difference was particularly large in training:
17.96% late among excluded orders versus 7.27% among retained orders.
Validation rates were 4.90% versus 3.90%.

The cutoff therefore changed the outcome distribution. It was
not simply removing a representative sample of orders.

Earlier model results are preliminary and apply to those filtered
populations. This audit does not establish how correcting the
evaluation will change model performance.

In [28]:
demo_orders = prepared_orders.loc[
    X_test.index,
    ["order_id", "order_approved_at"] + feature_columns + ["is_late"]
].copy()

# Generate scores using the saved model
demo_orders["risk_score"] = loaded_bundle["pipeline"].predict_proba(
    demo_orders[feature_columns]
)[:, 1]

demo_dir = project_dir / "data" / "processed"
demo_dir.mkdir(parents=True, exist_ok=True)

demo_path = demo_dir / "demo_orders.csv"
demo_orders.to_csv(demo_path, index=False)

print(f"Saved {len(demo_orders):,} demo orders.")
print(f"File: {demo_path}")

Saved 12,516 demo orders.
File: /Users/javiermorande/Desktop/olist-delivery-risk/data/processed/demo_orders.csv
